# 02 Colour-Space RGB+HSV Experiment

This notebook handles the colour-space side of the project. It converts the RGB synthetic dataset into six-channel RGB+HSV TIFFs, runs classical colour-space baselines, trains the experimental RGB+HSV YOLO26 segmentation model, and evaluates it on the test split.

Run `01_synthetic_generation.ipynb` before this notebook.


## Setup

The RGB+HSV dataset is derived from `data/synthetic/rgb`, so it uses the same examples and labels as the RGB dataset.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from naartjie_cv.config import class_names, load_config, project_root, resolve_path

ROOT = project_root()
CONFIG = ROOT / 'config' / 'project.yaml'
cfg = load_config(CONFIG)

print(f'Project root: {ROOT}')
print(f'Config: {CONFIG}')


## Prerequisite Check

This confirms that the RGB synthetic dataset exists before creating the multispectral version.


In [ ]:
rgb_data_yaml = resolve_path(cfg, cfg['paths']['synthetic_dir']) / 'data.yaml'
if not rgb_data_yaml.exists():
    raise FileNotFoundError(f'RGB synthetic dataset not found: {rgb_data_yaml}. Run 01_synthetic_generation.ipynb first.')
print(f'Found RGB dataset: {rgb_data_yaml}')


## Create RGB+HSV Multispectral Dataset

Ultralytics supports multispectral datasets via TIFF files and `channels` in the dataset YAML. This creates six-channel TIFFs in CHW order:

`R, G, B, H, S, V`


In [ ]:
from naartjie_cv.multispectral import create_rgb_hsv_dataset

rgb_hsv_summary = create_rgb_hsv_dataset(CONFIG)
print(json.dumps(rgb_hsv_summary, indent=2))


## Classical Colour-Space Baselines

These baselines provide non-deep-learning comparison points:

- HSV thresholding + morphology + contours
- L*a*b* + K-means + contours
- HSV/Lab candidate masks + colour/texture classifier

Reports are saved under `runs/classical`.


In [ ]:
from naartjie_cv.classical import run_classical_baselines

classical_reports = run_classical_baselines(CONFIG, split='test')
print(json.dumps(classical_reports, indent=2))


## Runtime Check

This verifies the active conda environment, PyTorch, CUDA, GPU, and Ultralytics installation before RGB+HSV YOLO training.


In [ ]:
from naartjie_cv.runtime import assert_training_runtime, print_runtime_report

report = assert_training_runtime()
print_runtime_report(report)


## Train RGB+HSV YOLO26

This trains the experimental six-channel model using `data/synthetic/rgb_hsv/data.yaml`. Keep the RGB model as the deployment fallback unless this path exports and validates cleanly.


In [ ]:
from naartjie_cv.train import train_yolo

rgb_hsv_results = train_yolo(CONFIG, variant='rgb_hsv')
rgb_hsv_results


## Test RGB+HSV Model

This evaluates the best RGB+HSV weights on the test split and writes JSON/CSV reports under `runs/evaluations`.


In [ ]:
from naartjie_cv.evaluate import evaluate_model

rgb_hsv_weights = ROOT / 'runs' / 'segment' / 'naartjie_rgb_hsv' / 'weights' / 'best.pt'
if not rgb_hsv_weights.exists():
    raise FileNotFoundError(f'RGB+HSV weights not found: {rgb_hsv_weights}')

rgb_hsv_report = evaluate_model(CONFIG, rgb_hsv_weights, variant='rgb_hsv', split='test')
print(json.dumps(rgb_hsv_report, indent=2))
